In [1]:
import random

# نفس بيانات المشكلة بتاعتنا
bin_capacity = 8
items = [1, 2, 3, 4, 1, 2, 4, 3, 5, 3, 4, 4, 5, 1, 5, 4, 4, 6, 2, 7, 4]

# 1. دالة التعبئة (First-Fit): بتاخد ترتيب معين للعناصر وتحطهم في الصناديق
def first_fit(chromosome):
    bins = []
    for item in chromosome:
        placed = False
        for b in bins:
            if sum(b) + item <= bin_capacity:
                b.append(item)
                placed = True
                break
        if not placed:
            bins.append([item])
    return bins

# 2. دالة الكفاءة (Fitness Function): بتقيم التوزيعة
# كل ما عدد الصناديق يقل، والصناديق تكون مليانة للاخر، الـ Fitness بيزيد
def calculate_fitness(bins):
    fitness = 0
    for b in bins:
        fill_ratio = sum(b) / bin_capacity
        fitness += (fill_ratio ** 2) # بنربع النسبة عشان نكافئ الصناديق المليانة أكتر
    return fitness / len(bins)

# 3. إنشاء الجيل الأول (Population)
def create_population(pop_size):
    population = []
    for _ in range(pop_size):
        # بنعمل ترتيب عشوائي (Shuffle) للعناصر عشان نعمل "كروموسوم" جديد
        shuffled_items = items.copy()
        random.shuffle(shuffled_items)
        population.append(shuffled_items)
    return population

# نجرب نعمل جيل أول مكون من 5 حلول عشوائية ونشوف كفاءتهم
POPULATION_SIZE = 5
current_population = create_population(POPULATION_SIZE)

print("=== تقييم الجيل الأول (حلول عشوائية) ===")
for i, chromosome in enumerate(current_population):
    bins = first_fit(chromosome)
    fitness_score = calculate_fitness(bins)
    print(f"حل رقم {i+1}: استخدم {len(bins)} صناديق | الكفاءة (Fitness): {fitness_score:.4f}")

=== تقييم الجيل الأول (حلول عشوائية) ===
حل رقم 1: استخدم 10 صناديق | الكفاءة (Fitness): 0.8781
حل رقم 2: استخدم 10 صناديق | الكفاءة (Fitness): 0.8719
حل رقم 3: استخدم 10 صناديق | الكفاءة (Fitness): 0.8688
حل رقم 4: استخدم 10 صناديق | الكفاءة (Fitness): 0.8719
حل رقم 5: استخدم 10 صناديق | الكفاءة (Fitness): 0.8781


In [3]:
import random
import time

# 4. التزاوج (Crossover): نسخة معدلة بتدعم الأرقام المتكررة (Duplicates)
def crossover(parent1, parent2):
    size = len(parent1)
    start, end = sorted(random.sample(range(size), 2))
    child = [-1] * size
    
    # 1. ناخد جزء عشوائي من الأب الأول
    child[start:end] = parent1[start:end]
    
    # 2. نجهز العناصر اللي لسة ناقصة من الأب التاني
    remaining_items = parent2.copy()
    for item in child[start:end]:
        remaining_items.remove(item) # بنحذف النسخ اللي خدناها خلاص
        
    # 3. نكمل الباقي في الأماكن الفاضية في الـ child
    rem_idx = 0
    for i in range(size):
        if child[i] == -1:
            child[i] = remaining_items[rem_idx]
            rem_idx += 1
            
    return child

# 5. الطفرة (Mutation): تبديل مكان عنصرين (Swap)
def mutate(chromosome, mutation_rate=0.1):
    if random.random() < mutation_rate:
        idx1, idx2 = random.sample(range(len(chromosome)), 2)
        chromosome[idx1], chromosome[idx2] = chromosome[idx2], chromosome[idx1]
    return chromosome

# 6. اختيار الأهل (Selection): بنختار 3 عشوائيين وناخد أحسنهم يتجوز
def select_parent(population, fitnesses, k=3):
    selected_indices = random.sample(range(len(population)), k)
    best_idx = max(selected_indices, key=lambda i: fitnesses[i])
    return population[best_idx]

# 7. الحلقة الرئيسية للـ Genetic Algorithm
def genetic_algorithm(pop_size=50, generations=100, mutation_rate=0.1):
    population = create_population(pop_size)
    best_overall_fitness = 0
    best_overall_bins = []
    
    for gen in range(generations):
        fitnesses = [calculate_fitness(first_fit(chrom)) for chrom in population]
        
        # حفظ أحسن حل في الجيل الحالي
        current_best_idx = fitnesses.index(max(fitnesses))
        if fitnesses[current_best_idx] > best_overall_fitness:
            best_overall_fitness = fitnesses[current_best_idx]
            best_overall_bins = first_fit(population[current_best_idx])
        
        new_population = []
        # بناء الجيل الجديد
        for _ in range(pop_size // 2):
            p1 = select_parent(population, fitnesses)
            p2 = select_parent(population, fitnesses)
            
            child1 = mutate(crossover(p1, p2), mutation_rate)
            child2 = mutate(crossover(p2, p1), mutation_rate)
            
            new_population.extend([child1, child2])
            
        population = new_population
        
    return best_overall_bins, best_overall_fitness

# نشغل الخوارزمية الجينية ونشوف التطور!
print("جاري تشغيل الخوارزمية الجينية (Evolution in progress)...")
start_time = time.time()

best_ga_bins, best_ga_fitness = genetic_algorithm(pop_size=50, generations=100, mutation_rate=0.1)

end_time = time.time()

print("-" * 40)
print(f"أحسن عدد صناديق وصلنا له بالـ GA: {len(best_ga_bins)}")
print(f"أحسن كفاءة (Fitness): {best_ga_fitness:.4f}")
print(f"الوقت المستغرق: {end_time - start_time:.4f} ثانية")
print("تفاصيل التوزيعة:")
for i, b in enumerate(best_ga_bins):
    print(f"صندوق {i+1}: {b} (مجموعهم {sum(b)} من أصل {bin_capacity})")

جاري تشغيل الخوارزمية الجينية (Evolution in progress)...
----------------------------------------
أحسن عدد صناديق وصلنا له بالـ GA: 10
أحسن كفاءة (Fitness): 0.9062
الوقت المستغرق: 0.0691 ثانية
تفاصيل التوزيعة:
صندوق 1: [4, 4] (مجموعهم 8 من أصل 8)
صندوق 2: [5, 1, 2] (مجموعهم 8 من أصل 8)
صندوق 3: [5, 3] (مجموعهم 8 من أصل 8)
صندوق 4: [4, 4] (مجموعهم 8 من أصل 8)
صندوق 5: [4, 4] (مجموعهم 8 من أصل 8)
صندوق 6: [7, 1] (مجموعهم 8 من أصل 8)
صندوق 7: [3, 4, 1] (مجموعهم 8 من أصل 8)
صندوق 8: [5, 3] (مجموعهم 8 من أصل 8)
صندوق 9: [6, 2] (مجموعهم 8 من أصل 8)
صندوق 10: [2] (مجموعهم 2 من أصل 8)
